## Practical-4  Implement Keras MLP for Multiclass Classification

**Domain:** Finance — predicting a simple trading signal (**Buy / Hold / Sell**)
from two basic indicators, using a small Multi-Layer Perceptron (MLP) built
with Keras.

In [1]:
# Import libraries
import os
os.environ["TF_DETERMINISTIC_OPS"] = "1"   # ask TensorFlow to use
                                            # deterministic op implementations

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical

# Fix random seeds so weight initialization and training are
# reproducible -- with a dataset this small, a different random
# starting point can change how quickly (or well) the model learns.
np.random.seed(42)
tf.random.set_seed(42)
tf.keras.utils.set_random_seed(42)

# ---------------------------------------------------------
# Example dataset: Trading Signal (Buy / Hold / Sell)
#
# Features (kept simple, on purpose):
#   Daily Return (%) -> how much the price moved today
#   RSI               -> momentum indicator (0-100)
#
# Label (3 classes):
#   0 -> Sell  (falling price, overbought)
#   1 -> Hold  (no strong signal either way)
#   2 -> Buy   (rising price, oversold / recovering)
#
# This is a small, hand-picked dataset -- just enough to
# show how a Keras MLP is built and trained for MULTICLASS
# classification (3 output classes instead of 2).
# ---------------------------------------------------------

X = np.array([
    [ 2.5, 22],   # big rise, oversold          -> Buy
    [ 1.8, 28],   # rise, low RSI                -> Buy
    [ 3.0, 20],   # big rise, very oversold      -> Buy

    [ 0.2, 50],   # flat, neutral RSI            -> Hold
    [-0.2, 48],   # flat, neutral RSI            -> Hold
    [ 0.4, 52],   # flat, neutral RSI            -> Hold

    [-2.0, 78],   # big drop, overbought         -> Sell
    [-1.5, 82],   # drop, overbought             -> Sell
    [-2.5, 85],   # big drop, very overbought    -> Sell
], dtype=np.float32)

y = np.array([2, 2, 2, 1, 1, 1, 0, 0, 0])   # 0=Sell, 1=Hold, 2=Buy

# Keras needs the labels ONE-HOT ENCODED for multiclass
# classification, e.g. class 2 (Buy) -> [0, 0, 1]
y_categorical = to_categorical(y, num_classes=3)

print("Features (Daily Return %, RSI):\n", X)
print("\nOne-hot labels:\n", y_categorical)

I0000 00:00:1787032514.610932   11859 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787032514.611626   11859 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1787032514.676737   11859 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1787032516.554863   11859 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787032516.555329   11859 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Features (Daily Return %, RSI):
 [[ 2.5 22. ]
 [ 1.8 28. ]
 [ 3.  20. ]
 [ 0.2 50. ]
 [-0.2 48. ]
 [ 0.4 52. ]
 [-2.  78. ]
 [-1.5 82. ]
 [-2.5 85. ]]

One-hot labels:
 [[0. 0. 1.]
 [0. 0. 1.]
 [0. 0. 1.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]]


In [2]:
# ---------------------------------------------------------
# Feature Scaling (Standardization)
#
# Daily Return (roughly -3 to +3) and RSI (roughly 0-100)
# are on very different scales. Without scaling, RSI would
# dominate the network's early learning simply because its
# raw numbers are much bigger -- not because it is actually
# more important.
#
# We standardize each feature to have mean 0 and standard
# deviation 1 so both features contribute fairly.
# ---------------------------------------------------------

X_mean = X.mean(axis=0)
X_std  = X.std(axis=0)

X_scaled = (X - X_mean) / X_std

print("Feature mean:", X_mean)
print("Feature std :", X_std)
print("\nScaled features:\n", X_scaled)

Feature mean: [ 0.18888892 51.666668  ]
Feature std : [ 1.8531921 24.       ]

Scaled features:
 [[ 1.2470974  -1.2361112 ]
 [ 0.86937076 -0.98611116]
 [ 1.5169021  -1.3194445 ]
 [ 0.00599564 -0.0694445 ]
 [-0.20984815 -0.15277784]
 [ 0.11391754  0.01388884]
 [-1.1811452   1.0972222 ]
 [-0.9113405   1.2638888 ]
 [-1.45095     1.3888888 ]]


In [3]:
# Build MLP model
model = Sequential([
    Dense(8, input_dim=2, activation='relu'),   # 8 = number of neurons
                                                 # More neurons -> learns more complex patterns
                                                 # Too many neurons -> more computation and possible overfitting
                                                 # ReLU helps learn non-linear relationships and trains faster

    Dense(4, activation='relu'),

    Dense(3, activation='softmax')              # 3 output neurons -- one per class (Sell/Hold/Buy)
                                                 # Softmax turns outputs into probabilities that sum to 1
                                                 # Used for MULTICLASS classification (more than 2 classes)
])

model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1787032518.490613   11859 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 8)              │            24 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            15 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 75 (300.00 B)

 Trainable params: 75 (300.00 B)

 Non-trainable params: 0 (0.00 B)

In [4]:
# Compile model
model.compile(
    optimizer='adam',          # Adam adapts learning rate automatically
                               # Good default optimizer
                               # Alternatives: SGD, RMSprop, AdamW

    loss='categorical_crossentropy',   # Used for multiclass classification
                                        # (binary_crossentropy is only for 2 classes)

    metrics=['accuracy']
)

In [5]:
# Train model
model.fit(X_scaled, y_categorical, epochs=300, verbose=1)

Epoch 1/300


E0000 00:00:1787032518.675431   11859 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.3333 - loss: 1.1045


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.3333 - loss: 1.1045


Epoch 2/300


...


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 1.0000 - loss: 0.2035


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 1.0000 - loss: 0.2035


Epoch 300/300



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 1.0000 - loss: 0.2021


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 1.0000 - loss: 0.2021


In [6]:
print("\nPredictions:")

class_names = ["Sell", "Hold", "Buy"]
predictions = model.predict(X_scaled)

for i in range(len(X)):
    predicted_class = np.argmax(predictions[i])   # index of the highest probability
    print(f"Input (Return%, RSI): {X[i]} => Probabilities: {np.round(predictions[i], 3)} "
          f"=> Predicted: {class_names[predicted_class]}")
    # argmax picks the class with the highest predicted probability
    # e.g. [0.05, 0.10, 0.85] => class 2 => "Buy"
    # Note: we predict using X_scaled (the same scaling used
    # for training) but print the original, human-readable
    # X values for clarity.


Predictions:



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step


Input (Return%, RSI): [ 2.5 22. ] => Probabilities: [0.012 0.042 0.947] => Predicted: Buy
Input (Return%, RSI): [ 1.8 28. ] => Probabilities: [0.033 0.115 0.852] => Predicted: Buy
Input (Return%, RSI): [ 3. 20.] => Probabilities: [0.006 0.022 0.972] => Predicted: Buy
Input (Return%, RSI): [ 0.2 50. ] => Probabilities: [0.18  0.668 0.152] => Predicted: Hold
Input (Return%, RSI): [-0.2 48. ] => Probabilities: [0.174 0.665 0.162] => Predicted: Hold
Input (Return%, RSI): [ 0.4 52. ] => Probabilities: [0.186 0.654 0.16 ] => Predicted: Hold
Input (Return%, RSI): [-2. 78.] => Probabilities: [0.88  0.091 0.028] => Predicted: Sell
Input (Return%, RSI): [-1.5 82. ] => Probabilities: [0.878 0.093 0.029] => Predicted: Sell
Input (Return%, RSI): [-2.5 85. ] => Probabilities: [0.934 0.052 0.014] => Predicted: Sell


E0000 00:00:1787032533.055759   11859 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


In [7]:
# -----------------------------
# Applications of MLP
# -----------------------------

# 1. Image Recognition
# A grayscale image is converted into pixel values.
# Example: A 28 x 28 image becomes 784 input values after flattening.
# These inputs are fed into the MLP.
# Hidden layers learn patterns from the pixels and the output layer predicts
# the image class (e.g., digit 0-9).
# Limitation:
# Since the image is flattened into a 1D vector, the MLP cannot understand
# the spatial relationship between neighbouring pixels.
# Hence, CNNs are generally preferred for image recognition tasks.

# 2. Finance
# MLP can be used to predict trading signals (Buy/Hold/Sell), credit risk,
# loan default, fraud detection and market trends using historical or
# engineered features -- as shown in this notebook.
# However, financial data has a low signal-to-noise ratio and market
# relationships change over time (non-stationary).
# A plain MLP can easily memorize historical noise instead of learning
# meaningful patterns, leading to overfitting.
# Therefore, feature engineering, regularization and proper time-series
# validation are important when using MLPs for financial applications.